Create ppi human graph

In [2]:
df = pd.read_csv('bio-pathways-network.csv')

edges = list(zip(df['Gene ID 1'], df['Gene ID 2']))

G_human = nx.Graph()
G_human.add_edges_from(edges)

print("Human nodes:", len(G_human.nodes()), "edges:", len(G_human.edges()))

Human nodes: 21557 edges: 342353


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
# import torch
# import torch.nn.functional as F
# from torch_geometric.nn import GraphConv
# from torch.nn import Linear
# from torch_geometric.nn import global_mean_pool
import urllib.request
# from nodevectors import Node2Vec

ModuleNotFoundError: No module named 'nodevectors'

Retrieve and convert yeast file to graph

In [5]:
urllib.request.urlretrieve(
    'http://snap.stanford.edu/deepnetbio-ismb/ipynb/yeast.edgelist',
    'yeast.edgelist'
)
yeast_file = "yeast.edgelist"
G_yeast = nx.read_edgelist(yeast_file)

print("Yeast nodes:", len(G_yeast.nodes()), "edges:", len(G_yeast.edges()))

Yeast nodes: 6526 edges: 532180


Graph Embedding, Labelling and feature extraction

In [ ]:
def create_labels(G, threshold=5):
    labels = {}
    for node, deg in dict(G.degree()).items():
        labels[node] = 1 if deg >= threshold else 0
    return labels

labels_yeast = create_labels(G_yeast, threshold=5)
labels_human = create_labels(G_human, threshold=10)

#print graph label
print("Yeast labels:", list(labels_yeast.items())[:10])
print("Human labels:", list(labels_human.items())[:10])

Yeast labels: [('YLR418C', 1), ('YOL145C', 1), ('YOR123C', 1), ('YBR279W', 1), ('YML069W', 1), ('YGL244W', 1), ('YGL207W', 1), ('YER164W', 1), ('YIL035C', 1), ('YOR061W', 1)]
Human labels: [(1394, 1), (2778, 1), (6331, 1), (17999, 1), (122704, 1), (54460, 1), (2597, 1), (2911, 1), (4790, 1), (79155, 1)]


In [8]:
g2v = Node2Vec(n_components=64, walklen=30, epochs=10)
g2v.fit(G_human)

AttributeError: module 'networkx' has no attribute 'adj_matrix'

DBSCAN

In [8]:
#!/usr/bin/env python3
"""
DBSCAN on Yeast & Human PPI (no node2vec/gensim)
------------------------------------------------
- Reads yeast.edgelist and PP-Pathways_ppi.csv
- Builds SVD embeddings from adjacency (fast & dependency-light)
- Sweeps DBSCAN hyperparams, picks best by silhouette
- Saves labeled embeddings and grid results as CSVs
"""

import argparse
import os
import sys
import warnings
from typing import List, Tuple

import numpy as np
import pandas as pd
import networkx as nx

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

# --------- Utils ---------
def eprint(*args, **kwargs):
    print(*args, file=sys.stderr, **kwargs)

def check_file_exists(path: str, desc: str):
    if not os.path.isfile(path):
        eprint(f"[ERROR] {desc} not found at: {path}")
        sys.exit(1)

# --------- Load graphs ---------
def load_yeast_graph(path: str) -> nx.Graph:
    G = nx.read_edgelist(path)
    print(f"[Yeast] nodes={G.number_of_nodes()} edges={G.number_of_edges()}")
    return G

def load_human_graph(path: str) -> nx.Graph:
    df = pd.read_csv(path)
    # Try common PPI column names first; else use the first two columns
    candidates = [
        ("protein1", "protein2"),
        ("Protein1", "Protein2"),
        ("prot1", "prot2"),
        ("u", "v"),
        ("source", "target"),
        ("from", "to"),
    ]
    ucol = vcol = None
    for a, b in candidates:
        if a in df.columns and b in df.columns:
            ucol, vcol = a, b
            break
    if ucol is None:
        # Fallback — first two columns
        ucol, vcol = df.columns[:2]
        print(f"[Human] Using columns '{ucol}' and '{vcol}' as edges (adjust if needed).")
    edges = list(zip(df[ucol].astype(str), df[vcol].astype(str)))
    G = nx.Graph()
    G.add_edges_from(edges)
    print(f"[Human] nodes={G.number_of_nodes()} edges={G.number_of_edges()}")
    return G

# --------- Embeddings via SVD ---------
def svd_embeddings(
    G: nx.Graph,
    dimensions: int = 64,
    random_state: int = 0,
    keep_lcc: bool = True
) -> Tuple[pd.DataFrame, np.ndarray, List[str]]:
    """
    Returns:
      emb_df: DataFrame [n_nodes x (dimensions + node)]
      Xs: scaled embedding for clustering
      nodes: node order
    """
    if keep_lcc and not nx.is_empty(G):
        if not nx.is_connected(G):
            lcc_nodes = max(nx.connected_components(G), key=len)
            G = G.subgraph(lcc_nodes).copy()
            print(f"[SVD] Kept largest component: nodes={G.number_of_nodes()} edges={G.number_of_edges()}")

    if G.number_of_nodes() == 0:
        raise ValueError("Graph has no nodes after preprocessing.")

    nodes = list(G.nodes())
    # Sparse adjacency as CSR
    A = nx.to_scipy_sparse_array(G, nodelist=nodes, dtype=np.float32, format="csr")

    d = min(dimensions, min(A.shape) - 1)
    d = max(d, 2)  # ensure >=2
    svd = TruncatedSVD(n_components=d, random_state=random_state)
    X = svd.fit_transform(A)

    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    emb_df = pd.DataFrame(X, columns=[f"f{i}" for i in range(d)])
    emb_df["node"] = nodes
    return emb_df, Xs, nodes

# --------- K-distance elbow (optional) ---------
def k_distance_plot(Xs: np.ndarray, k: int = 10, title: str = "k-distance (elbow ~ eps)"):
    nbrs = NearestNeighbors(n_neighbors=k).fit(Xs)
    dists, _ = nbrs.kneighbors(Xs)
    kd = np.sort(dists[:, -1])
    plt.figure()
    plt.plot(kd)
    plt.title(title)
    plt.xlabel("Points sorted by distance")
    plt.ylabel(f"Distance to {k}-th NN")
    plt.tight_layout()
    plt.show()

# --------- DBSCAN grid ---------
def run_dbscan_grid(
    Xs: np.ndarray,
    eps_list: Tuple[float, ...] = (0.8, 1.0, 1.2, 1.5, 2.0, 3.0),
    min_samples_list: Tuple[int, ...] = (5, 10, 20, 30)
) -> Tuple[dict, pd.DataFrame]:
    results = []
    best = {"sil": -np.inf, "labels": None, "params": None, "n_clusters": 0, "noise_ratio": None}
    for eps in eps_list:
        for ms in min_samples_list:
            model = DBSCAN(eps=eps, min_samples=ms, n_jobs=-1)
            labels = model.fit_predict(Xs)
            n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
            noise = float(np.mean(labels == -1))
            if n_clusters >= 2:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    try:
                        sil = float(silhouette_score(Xs, labels))
                    except Exception:
                        sil = np.nan
            else:
                sil = np.nan
            results.append((eps, ms, n_clusters, noise, sil))
            if not np.isnan(sil) and sil > best["sil"]:
                best.update({
                    "sil": sil,
                    "labels": labels,
                    "params": (eps, ms),
                    "n_clusters": n_clusters,
                    "noise_ratio": noise
                })
    res_df = pd.DataFrame(results, columns=["eps", "min_samples", "n_clusters", "noise_ratio", "silhouette"])\
        .sort_values(["silhouette", "n_clusters"], ascending=[False, False]).reset_index(drop=True)
    return best, res_df

# --------- PCA scatter (optional) ---------
def pca_scatter(Xs: np.ndarray, labels: np.ndarray, title: str):
    if labels is None or len(set(labels)) <= 1:
        print(f"[Plot] {title}: skipping (no meaningful clusters).")
        return
    pca = PCA(n_components=2, random_state=0)
    X2 = pca.fit_transform(Xs)
    plt.figure()
    plt.scatter(X2[:, 0], X2[:, 1], s=8, alpha=0.85, c=labels)
    plt.title(title)
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.tight_layout()
    plt.show()

# --------- Main ---------
def main():
    ap = argparse.ArgumentParser(description="DBSCAN on Yeast & Human PPI using SVD embeddings")
    ap.add_argument("--yeast", default="yeast.edgelist", help="Path to yeast edgelist file")
    ap.add_argument("--human", default="PP-Pathways_ppi.csv", help="Path to human PPI CSV")
    ap.add_argument("--dims", type=int, default=64, help="Embedding dimension (SVD)")
    ap.add_argument("--seed", type=int, default=0, help="Random seed")
    ap.add_argument("--eps", type=str, default="0.8,1.0,1.2,1.5,2.0,3.0", help="Comma-separated eps values")
    ap.add_argument("--min_samples", type=str, default="5,10,20,30", help="Comma-separated min_samples values")
    ap.add_argument("--plots", action="store_true", help="Show PCA cluster plots")
    ap.add_argument("--kdist", action="store_true", help="Show k-distance elbow plots (k=10)")
    ap.add_argument("--outdir", default=".", help="Directory to save CSV outputs")
    args = ap.parse_args()

    check_file_exists(args.yeast, "Yeast edgelist")
    check_file_exists(args.human, "Human PPI CSV")

    eps_list = tuple(float(x.strip()) for x in args.eps.split(",") if x.strip())
    ms_list = tuple(int(x.strip()) for x in args.min_samples.split(",") if x.strip())

    # Load graphs
    G_yeast = load_yeast_graph(args.yeast)
    G_human = load_human_graph(args.human)

    # Embeddings
    embY_df, XsY, nodesY = svd_embeddings(G_yeast, dimensions=args.dims, random_state=args.seed)
    embH_df, XsH, nodesH = svd_embeddings(G_human, dimensions=args.dims, random_state=args.seed)
    print(f"[Embeddings] Yeast: {embY_df.shape}, Human: {embH_df.shape}")

    # Optional k-distance elbow
    if args.kdist:
        k_distance_plot(XsY, k=10, title="Yeast: k-distance (k=10)")
        k_distance_plot(XsH, k=10, title="Human: k-distance (k=10)")

    # DBSCAN grid
    bestY, gridY = run_dbscan_grid(XsY, eps_list=eps_list, min_samples_list=ms_list)
    bestH, gridH = run_dbscan_grid(XsH, eps_list=eps_list, min_samples_list=ms_list)

    # Reports
    def report(name, best, grid):
        print(f"\n=== {name}: DBSCAN results ===")
        if best["params"] is None:
            print("No configuration produced >= 2 clusters. Try larger eps or smaller min_samples.")
        else:
            eps, ms = best["params"]
            print(f"Best params: eps={eps}, min_samples={ms}")
            print(f"Silhouette: {best['sil']:.4f}")
            print(f"#Clusters (excl. noise): {best['n_clusters']}")
            print(f"Noise ratio: {best['noise_ratio']:.3f}")
        print("\nTop candidates:")
        print(grid.head(10).to_string(index=False))

    report("Yeast", bestY, gridY)
    report("Human", bestH, gridH)

    # Attach labels
    embY_df["dbscan_label"] = bestY["labels"] if bestY["labels"] is not None else -1
    embH_df["dbscan_label"] = bestH["labels"] if bestH["labels"] is not None else -1

    # Plots
    if args.plots:
        pca_scatter(XsY, bestY["labels"], "Yeast: DBSCAN clusters (PCA)")
        pca_scatter(XsH, bestH["labels"], "Human: DBSCAN clusters (PCA)")

    # Save outputs
    os.makedirs(args.outdir, exist_ok=True)
    out_ye = os.path.join(args.outdir, "yeast_embeddings_dbscan.csv")
    out_hu = os.path.join(args.outdir, "human_embeddings_dbscan.csv")
    out_grid_y = os.path.join(args.outdir, "dbscan_grid_yeast.csv")
    out_grid_h = os.path.join(args.outdir, "dbscan_grid_human.csv")

    embY_df.to_csv(out_ye, index=False)
    embH_df.to_csv(out_hu, index=False)
    gridY.to_csv(out_grid_y, index=False)
    gridH.to_csv(out_grid_h, index=False)

    print(f"\nSaved:\n  {out_ye}\n  {out_hu}\n  {out_grid_y}\n  {out_grid_h}")

if __name__ == "__main__":
    import sys
    if "ipykernel" in sys.modules:
        # Running inside Jupyter or IPython — call main() manually with defaults
        sys.argv = ["", "--yeast", "yeast.edgelist", "--human", "bio-pathways-network.csv"]
    try:
        main()
    except KeyboardInterrupt:
        eprint("\n[Interrupted]")
        sys.exit(130)
    except Exception as e:
        eprint(f"[FATAL] {e}")
        raise


[Yeast] nodes=6526 edges=532180
[Human] Using columns 'Gene ID 1' and 'Gene ID 2' as edges (adjust if needed).
[Human] nodes=21557 edges=342353
[SVD] Kept largest component: nodes=21521 edges=342316
[Embeddings] Yeast: (6526, 65), Human: (21521, 65)

=== Yeast: DBSCAN results ===
Best params: eps=2.0, min_samples=5
Silhouette: -0.2234
#Clusters (excl. noise): 3
Noise ratio: 0.815

Top candidates:
 eps  min_samples  n_clusters  noise_ratio  silhouette
 2.0            5           3     0.815201   -0.223433
 1.2           20           2     0.883543   -0.281266
 1.2           10           3     0.879405   -0.281470
 1.2           30           2     0.885075   -0.282774
 1.2            5           4     0.876494   -0.288198
 1.0           20           2     0.895648   -0.289537
 1.0           10           4     0.888293   -0.290003
 1.0            5           7     0.884462   -0.297610
 0.8           30           2     0.905302   -0.304042
 0.8           10           5     0.897027   -0.30